In [ ]:
import jax
import os

import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

import pulse_simulator as ps

In [2]:
from datetime import datetime
import numpy as np
import pickle as pkl

import qiskit as qk
from qiskit.quantum_info import Operator
from qiskit import pulse
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d

from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm, logm
from tqdm import tqdm

import matplotlib.pyplot as plt

In [ ]:
# Config stuff
jax.config.update("jax_enable_x64", True)

# Set the memory fraction
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="0.49"

# jax.config.update("jax_platform_name", "cpu")
qk_d.array.Array.set_default_backend("jax")

# Check if GPU is available
jax.default_backend()

# Pulses

In [4]:
num_qubits = 10

coupling_map = (
    [[i, i + 1] for i in range(num_qubits - 1)] 
    + [[i + 1, i] for i in range(num_qubits - 1)]
)

backend = qk_fp.GenericBackendV2(
    num_qubits=num_qubits,
    basis_gates=["id", "rz", "sx", "x", "ecr"],
    coupling_map=coupling_map,
    seed=17,
)

# backend = qk_fp.FakeMumbai()
# backend = qk_fp.FakeManila()

In [5]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

dt = 0.5 # backend.configuration().dt * ns
desired_1qb_duration = 120  # dt
desired_2qb_duration = 120 * 4  # dt

# Gate time in SI units (granularity restriction)
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_duration * dt / ns, backend)
duration_2qb = ps.qiskit_closest_multiple_of_16(desired_2qb_duration * dt / ns, backend)

# Pulses

This section should be re-run with the simulator, as needed.

In [ ]:
# 1qb shapes
# ==========
fname_1qb = "../pulse_simulator/gatesets/controls_v3_T_120_dt_0.5_a_0.35_dda_0.03.csv"
pico_a = ps.from_csv(fname_1qb, delimiter="\t")
pico_a = np.vstack([pico_a, 1.0 * np.ones(pico_a.shape[1])])

names_1qb = ["X1", "X2", "SX1", "SX2", "ID1", "ID2"]
angles_1qb = [np.pi / 2, np.pi / 2, np.pi / 4, np.pi / 4, np.pi, 0.0]

cal_1qb_robust = ps.calibrate_gateset(
    gateset=pico_a,
    names=names_1qb,
    angles=angles_1qb,
    duration=duration_1qb,
    dt=dt,
    samples=60,
)

# defaults
cal_1qb_default = ps.default_single_qubit_gateset(
    names_1qb,
    duration_1qb,
    dt,
)

# 2qb shapes
# ==========
# ECR gate
model_name = "SWPT"
angle = ps.two_qubit_models.calibrate_default_cross_resonance_drive(model_name)
gates_2qb = {"+": angle, "-": -angle}
cal_2qb_default = ps.default_gateset(gates_2qb, duration_2qb, dt)

# Circuits

### Ising

In [7]:
registers = [0, 1, 2, 3]
ising = ps.TransverseIsingModel(quantum_registers=registers)
# circ = ising.second_order_Trotter_unitary()
circ = ising.first_order_Trotter_unitary()
# circ.draw()


In [ ]:
simulator = ps.Simulator(registers, backend)
circ1 = simulator.build_circuit(moments=simulator.build_moments(circ), insert_barriers=True)
circ1.draw("mpl")

In [9]:
new_moments = []
for moment in simulator.build_moments(circ):
    if ps.get_moment_case(moment) == "ecr":
        continue
    else:
        new_moments.append(moment)

circ2 = simulator.build_circuit(moments=new_moments, insert_barriers=True)

In [ ]:
# circ1.draw('mpl')
circ2.draw('mpl')

### Random Circuit

In [11]:
from qiskit.circuit.random import random_circuit

In [ ]:
registers = [0, 1]
simulator = ps.Simulator(registers, backend)

depth = 10
rand_circ = random_circuit(len(registers), 17, max_operands=1)
circ3 = simulator.build_circuit(moments=simulator.build_moments(rand_circ), insert_barriers=True)
circ3.draw('mpl')

### Simulataneous XY4

### RB Circuit [[link](https://qiskit-community.github.io/qiskit-experiments/manuals/verification/randomized_benchmarking.html)]

In [8]:
from qiskit_experiments.library import StandardRB

In [ ]:
registers = [0, 1]#, 2] #, 3]
seeds = [17, 18]#, 19] #, 20]
simulator = ps.Simulator(registers, backend)
simult_rb = qk.QuantumCircuit(len(registers))

for r, seed in zip(registers, seeds):
    # Create an RB experiment
    lengths = [10]
    rb_exp = StandardRB([0], lengths, num_samples=1, seed=seed)
    rb_circ = qk.transpile(rb_exp.circuits()[0], backend, optimization_level=1)

    rb_circ = ps.remove_final_measurement(ps.remove_idle_wires(rb_circ))
    # rb_circ.draw(output="mpl")
    simult_rb.compose(rb_circ, qubits=r, inplace=True)

simult_rb = simulator.build_circuit(moments=simulator.build_moments(simult_rb), insert_barriers=True)
simult_rb.draw('mpl')

# Simulator

In this example, we loop over the one and two qubit gatesets.

You can make changes to the simulator's random variable params between runs, instead of changing the actual simulator variables.

This is because actually using the sampled values for crosstalk required thinking about frequency layouts.

In [14]:
# registers = [0, 1, 2, 3]
# circ = circ1

registers = [0, 1]
circ = simult_rb
simulator = ps.Simulator(registers, backend)

include_crosstalk = False
include_dissipation = True
model_name = "SWPT"

# --------------------------------

# modify pulse shapes, if needed
cal_1qb_robust = ps.calibrate_gateset(
    gateset=pico_a,
    names=names_1qb,
    angles=angles_1qb,
    duration=duration_1qb,
    dt=dt,
    samples=60,
    verbose=False,
)

cal_1qb_default = ps.default_single_qubit_gateset(
    names_1qb,
    duration_1qb,
    dt,
)

angle = ps.two_qubit_models.calibrate_default_cross_resonance_drive(model_name)
gates_2qb = {"+": angle, "-": -angle}
cal_2qb_default = ps.default_gateset(gates_2qb, duration_2qb, dt)

# --------------------------------
cases = [
    "1-default, 2-default",
    "1-robust, 2-default",
    # "1-default, 2-robust", 
    # "1-robust, 2-robust"
]
gateset_pairs = [
    [cal_1qb_default, cal_2qb_default],
    [cal_1qb_robust, cal_2qb_default],
    # [cal_1qb_default, cal_2qb_robust],
    # [cal_1qb_robust, cal_2qb_robust],
]
fidelities = {}

for case, pair in zip(cases, gateset_pairs):
    gateset1, gateset2 = pair
    res = simulator.simulate_circuit(
        circ,
        gateset1,
        gateset2,
        dt,
        include_crosstalk=include_crosstalk,
        include_dissipation=include_dissipation,
        cross_resonance_model_name=model_name,
        gpu=True,
    )

    moments = simulator.get_moments()

    measure_times = (
        np.cumsum(ps.durations_of_moments(moments, duration_1qb, duration_2qb)) * dt
    )

    U_goals = []
    for i in range(len(moments)):
        U_goals.append(ps.build_operator(moments[0 : i + 1], registers))

    # skip initial state in result TODO: should we even return it?
    if include_dissipation:
        fids = np.array(
            [
                qk.quantum_info.process_fidelity(R, qk.quantum_info.SuperOp(U))
                for R, U in zip(res[1:], U_goals)
            ]
        )
    else:
        fids = np.array([ps.operator_fidelity(R, U) for R, U in zip(res[1:], U_goals)])
    fidelities[case] = fids

In [ ]:
fig, ax = plt.subplots(1)

for case in cases:
    ax.plot(measure_times / duration_1qb, 1 - fidelities[case], label=case, marker="o")

# ax.set_yscale("log")
ax.set_xlabel("Layers (Elapsed time / 1QB duration)")
ax.set_ylabel("Infidelity")
ax.legend()